In [1]:
import math
import time
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


import torch
import torch.nn as nn


from datasets import sample_domain
from datasets import sample_TC
from datasets import sample_BC_S0
from datasets import sample_BC_Smax
from datasets import sample_domain
from datasets import sample_BC_Imax

SEED = 42
torch.manual_seed(SEED)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float32



In [4]:
import torch
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


S0        = 57.42
S_max     = 148.44
I_max     = 148.44
sigma     = 0.404
sigma_max = 0.807
r_max     = 0.5


Np   = 3000
n_bc = 1500
n_tc = 1500


X_dom = sample_domain(Np, S_max, I_max, r_max, sigma_max)

X_tc, Y_tc = sample_TC(n_tc, S_max, I_max, r_max, sigma_max)
X_bc_s0, Y_bc_s0 = sample_BC_S0(n_bc, S_max, I_max, r_max, sigma_max)
X_bc_smax, Y_bc_smax = sample_BC_Smax(n_bc, S_max, I_max, r_max, sigma_max)
X_bc_imax, Y_bc_imax = sample_BC_Imax(n_bc, S_max, I_max, r_max, sigma_max)




In [17]:
# DOMAIN visualization + basic statistics (PINN)

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


Xd = X_dom.detach().cpu().numpy()
df = pd.DataFrame(Xd, columns=["S", "I", "t", "r", "sigma"])


# BASIC STATISTICS 

print("DOMAIN: BASIC STATISTICS")
stats_df = df.agg(["min", "mean", "std", "max"])
print(stats_df)

# MARGINAL DISTRIBUTIONS (show domain ranges)


fig = go.Figure()
fig.add_trace(go.Histogram(x=df["S"], name="S", opacity=0.6))
fig.add_trace(go.Histogram(x=df["I"], name="I", opacity=0.6))
fig.add_trace(go.Histogram(x=df["t"], name="t", opacity=0.6))
fig.add_trace(go.Histogram(x=df["r"], name="r", opacity=0.6))
fig.add_trace(go.Histogram(x=df["sigma"], name="sigma", opacity=0.6))

fig.update_layout(
    title="Domain marginal distributions",
    barmode="overlay",
    template="plotly_white",
    xaxis_title="Value",
    yaxis_title="Count"
)
fig.show()


# 3D DOMAIN VISUALIZATION (geometric intuition)


fig = px.scatter_3d(
    df,
    x="S",
    y="I",
    z="t",
    opacity=0.5,
    title="Domain coverage in (S, I, t)"
)
fig.update_layout(template="plotly_white")
fig.show()




DOMAIN: BASIC STATISTICS
               S           I         t         r     sigma
min     0.061067    0.086601  0.000433  0.000053  0.000133
mean   74.746391   74.848396  0.503875  0.249639  0.403113
std    42.443340   42.756481  0.289014  0.143625  0.232949
max   148.419128  148.423203  0.999978  0.499947  0.806981


In [18]:
import numpy as np
import pandas as pd
import plotly.express as px

def make_df(X, Y):
    X = X.detach().cpu().numpy()
    Y = Y.detach().cpu().numpy().ravel()
    df = pd.DataFrame(X, columns=["S", "I", "t", "r", "sigma"])
    df["V"] = Y
    return df

df_tc   = make_df(X_tc, Y_tc)
df_s0   = make_df(X_bc_s0, Y_bc_s0)
df_smax = make_df(X_bc_smax, Y_bc_smax)
df_imax = make_df(X_bc_imax, Y_bc_imax)

# =========================
# TERMINAL CONDITION (t = 1)
# =========================

# Tutti i punti hanno t = 1 (maturità).
# S e I coprono l’intero dominio.
# V coincide con il payoff max(I − 1, 0).
print("TC statistics")
print(df_tc[["S","I","t","r","sigma","V"]].agg(["min","mean","std","max"]))

fig = px.scatter_3d(
    df_tc, x="S", y="I", z="t", opacity=0.6,
    title="Terminal condition: points lie on t = 1 hyperplane"
)
fig.update_layout(template="plotly_white")
fig.show()

fig = px.scatter(
    df_tc, x="I", y="V", opacity=0.6,
    title="Terminal condition: payoff shape V = max(I − 1, 0)"
)
fig.update_layout(template="plotly_white")
fig.show()

# =========================
# BOUNDARY CONDITION S = 0
# =========================

# Tutti i punti hanno S = 0.
# I e t variano nel dominio.
# V è il payoff scontato, quindi non cresce con S.
print("\nBC S = 0 statistics")
print(df_s0[["S","I","t","r","sigma","V"]].agg(["min","mean","std","max"]))

fig = px.scatter_3d(
    df_s0, x="S", y="I", z="t", opacity=0.6,
    title="Boundary S = 0: points lie on S = 0 plane"
)
fig.update_layout(template="plotly_white")
fig.show()

fig = px.scatter(
    df_s0, x="I", y="V", opacity=0.6,
    title="Boundary S = 0: discounted payoff vs I"
)
fig.update_layout(template="plotly_white")
fig.show()

# =========================
# BOUNDARY CONDITION S = S_max
# =========================

# Tutti i punti hanno S = S_max.
# I e t variano.
# V mostra crescita quasi lineare in S (regime asintotico).
print("\nBC S = S_max statistics")
print(df_smax[["S","I","t","r","sigma","V"]].agg(["min","mean","std","max"]))

fig = px.scatter_3d(
    df_smax, x="S", y="I", z="t", opacity=0.6,
    title="Boundary S = S_max: points lie on S = S_max plane"
)
fig.update_layout(template="plotly_white")
fig.show()

fig = px.scatter(
    df_smax, x="S", y="V", opacity=0.6,
    title="Boundary S = S_max: asymptotic linear growth of V"
)
fig.update_layout(template="plotly_white")
fig.show()

# =========================
# BOUNDARY CONDITION I = I_max
# =========================

# Tutti i punti hanno I = I_max.
# S e t variano.
# V cresce con S ed è coerente con il regime di media elevata.
print("\nBC I = I_max statistics")
print(df_imax[["S","I","t","r","sigma","V"]].agg(["min","mean","std","max"]))

fig = px.scatter_3d(
    df_imax, x="S", y="I", z="t", opacity=0.6,
    title="Boundary I = I_max: points lie on I = I_max plane"
)
fig.update_layout(template="plotly_white")
fig.show()

fig = px.scatter(
    df_imax, x="S", y="V", opacity=0.6,
    title="Boundary I = I_max: value vs S at large average"
)
fig.update_layout(template="plotly_white")
fig.show()


TC statistics
               S           I    t         r     sigma           V
min     0.015032    0.049848  1.0  0.000262  0.000549    0.000000
mean   73.822639   74.225464  1.0  0.254190  0.390690   73.230164
std    43.520187   43.250462  0.0  0.147197  0.237930   43.242458
max   148.345459  148.418442  1.0  0.499633  0.806792  147.418442



BC S = 0 statistics
        S           I         t         r     sigma           V
min   0.0    0.002168  0.002234  0.000662  0.001120    0.000000
mean  0.0   74.617149  0.502968  0.250039  0.410234   65.408508
std   0.0   42.446281  0.286323  0.142295  0.227727   38.394550
max   0.0  148.384476  0.999950  0.499705  0.805562  146.994720



BC S = S_max statistics
               S           I         t         r     sigma           V
min   148.440002    0.301043  0.001377  0.000789  0.001114    7.618269
mean  148.439972   75.722748  0.495824  0.250266  0.398319  135.174713
std     0.000000   43.128067  0.288476  0.145916  0.232200   51.979446
max   148.440002  148.340912  0.999439  0.499920  0.804769  274.076172



BC I = I_max statistics
               S           I         t         r     sigma           V
min     0.119931  148.440002  0.000290  0.000194  0.000028   98.226250
mean   73.341377  148.439972  0.497563  0.249814  0.403776  164.341141
std    43.441685    0.000000  0.284319  0.143937  0.235195   25.718027
max   148.389252  148.440002  0.997994  0.499025  0.806337  283.660583


In [20]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

def to_df(X, label):
    X = X.detach().cpu().numpy()
    df = pd.DataFrame(X, columns=["S", "I", "t", "r", "sigma"])
    df["type"] = label
    return df

df_tc   = to_df(X_tc, "TC: t = 1")
df_s0   = to_df(X_bc_s0, "BC: S = 0")
df_smax = to_df(X_bc_smax, "BC: S = S_max")
df_imax = to_df(X_bc_imax, "BC: I = I_max")

# Intersections
tc_s0   = df_tc[(df_tc["S"] == 0)]
tc_smax = df_tc[(df_tc["S"] == S_max)]
tc_imax = df_tc[(df_tc["I"] == I_max)]

s0_imax   = df_s0[(df_s0["I"] == I_max)]
smax_imax = df_smax[(df_smax["I"] == I_max)]

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=df_tc["S"], y=df_tc["I"], z=df_tc["t"],
    mode="markers", name="TC (t=1)",
    marker=dict(size=3, opacity=0.4)
))

fig.add_trace(go.Scatter3d(
    x=df_s0["S"], y=df_s0["I"], z=df_s0["t"],
    mode="markers", name="BC S=0",
    marker=dict(size=3, opacity=0.4)
))

fig.add_trace(go.Scatter3d(
    x=df_smax["S"], y=df_smax["I"], z=df_smax["t"],
    mode="markers", name="BC S=S_max",
    marker=dict(size=3, opacity=0.4)
))

fig.add_trace(go.Scatter3d(
    x=df_imax["S"], y=df_imax["I"], z=df_imax["t"],
    mode="markers", name="BC I=I_max",
    marker=dict(size=3, opacity=0.4)
))

fig.add_trace(go.Scatter3d(
    x=tc_s0["S"], y=tc_s0["I"], z=tc_s0["t"],
    mode="markers", name="TC ∩ S=0",
    marker=dict(size=6, color="red")
))

fig.add_trace(go.Scatter3d(
    x=tc_smax["S"], y=tc_smax["I"], z=tc_smax["t"],
    mode="markers", name="TC ∩ S=S_max",
    marker=dict(size=6, color="orange")
))

fig.add_trace(go.Scatter3d(
    x=tc_imax["S"], y=tc_imax["I"], z=tc_imax["t"],
    mode="markers", name="TC ∩ I=I_max",
    marker=dict(size=6, color="green")
))

fig.add_trace(go.Scatter3d(
    x=s0_imax["S"], y=s0_imax["I"], z=s0_imax["t"],
    mode="markers", name="S=0 ∩ I=I_max",
    marker=dict(size=6, color="purple")
))

fig.add_trace(go.Scatter3d(
    x=smax_imax["S"], y=smax_imax["I"], z=smax_imax["t"],
    mode="markers", name="S=S_max ∩ I=I_max",
    marker=dict(size=6, color="brown")
))

fig.update_layout(
    title="Intersections of Terminal and Boundary Conditions in (S, I, t)",
    scene=dict(
        xaxis_title="S",
        yaxis_title="I",
        zaxis_title="t"
    ),
    template="plotly_white"
)

fig.show()


In [19]:
import numpy as np
import pandas as pd
import plotly.express as px

def to_df(X, label):
    X = X.detach().cpu().numpy()
    df = pd.DataFrame(X, columns=["S", "I", "t", "r", "sigma"])
    df["type"] = label
    return df

df_tc   = to_df(X_tc, "TC: t=1")
df_s0   = to_df(X_bc_s0, "BC: S=0")
df_smax = to_df(X_bc_smax, "BC: S=S_max")
df_imax = to_df(X_bc_imax, "BC: I=I_max")

df = pd.concat([df_tc, df_s0, df_smax, df_imax], ignore_index=True)

# (S, I) — mostra pareti verticali/orizzontali e le intersezioni come punti/linee
fig = px.scatter(
    df, x="S", y="I", color="type", opacity=0.6,
    title="2D projection (S, I): BC and TC geometry"
)
fig.update_layout(template="plotly_white")
fig.show()

# (S, t) — TC è una linea orizzontale t=1; BC S=0/S_max sono linee verticali
fig = px.scatter(
    df, x="S", y="t", color="type", opacity=0.6,
    title="2D projection (S, t): TC and S-boundaries"
)
fig.update_layout(template="plotly_white")
fig.show()

# (I, t) — TC è t=1; BC I=I_max è una linea verticale
fig = px.scatter(
    df, x="I", y="t", color="type", opacity=0.6,
    title="2D projection (I, t): TC and I-boundary"
)
fig.update_layout(template="plotly_white")
fig.show()
